In [ ]:
# pip install pypdf2

In [ ]:
# pip install pdfplumbe

In [ ]:
# pip install pikepdf

In [ ]:
# pip install --upgrade pdfminer.six pdfplumber

In [ ]:
# pip install pywin32

In [ ]:
import pdfplumber
import re
import os
import time # For tracking execution time
import pandas as pd
import numpy as np
import pikepdf  # Library for PDF handling
import win32print
import win32api
from datetime import datetime
import fitz  # PyMuPDF

1. Extract Entry Number from PDF text

In [ ]:
def extract_flexible_pattern(text):
    # Pattern 1: space & hyphen  → A013 x-xxxx-xxxxx
    pattern_with_separator = r'A0\d{1,2}\s\d{1,4}-\d{1,4}-\d{1,5}'
    
    # Pattern 2: numbers all together → A003xxxxxxxxxx
    pattern_no_separator = r'A0\d{2}\d{10,13}'

    match = re.search(pattern_with_separator, text)
    if match:
        return match.group(0)
    
    match = re.search(pattern_no_separator, text)
    if match:
        return match.group(0)
    
    return None


def transform_pattern(pattern):
    if pattern:
        # delete space & hyphen → only number for both 2 patterns
        return pattern.replace(' ', '').replace('-', '')
    return pattern

def process_pdf_files_in_folder(folder_path):
    data = []
    
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.pdf'):
            file_path = os.path.join(folder_path, file_name)
            
            with pdfplumber.open(file_path) as pdf:
                if len(pdf.pages) > 0:
                    first_page = pdf.pages[0]
                    text = first_page.extract_text()
                    page_count = len(pdf.pages)
                    
                    # เก็บค่า original ก่อน transform
                    matched_data = extract_flexible_pattern(text)
                    transformed_data = transform_pattern(matched_data)
                    
                    data.append({
                        'File Name': file_name,
                        'Original Pattern': matched_data,
                        'Matched Data': transformed_data,
                        'Page Count': page_count
                    })
    
    df = pd.DataFrame(data)
    # order column
    df = df[['File Name', 'Original Pattern', 'Matched Data', 'Page Count']]
    
    return df

2. Load Excel Content into a pandas DataFrame

In [ ]:
# Define folder paths (PDF)
pdf_folder_path = r"<YOUR_PATH>"

# Start the timer
start_time = time.time()

# Process PDF files and get the DataFrame
pdf_df = process_pdf_files_in_folder(pdf_folder_path)

# End the timer
end_time = time.time()

# Calculate total execution time
execution_time = end_time - start_time
print(f"Total execution time: {execution_time:.2f} seconds")

# Print the DataFrames
print("PDF DataFrame:")
print(pdf_df)

3. Matching -> Calculate

In [ ]:
folder_path = r"<YOUR_PATH>" # Local Centen Excel file folder path
excel_files = [f for f in os.listdir(folder_path) if f.endswith(('.xlsx', '.xls'))]
dataframes = []
start_time = time.time()

for excel_file in excel_files:
    file_path = os.path.join(folder_path, excel_file)
    df = pd.read_excel(file_path)
    dataframes.append(df)
    print(f"Loaded {excel_file}")

if dataframes:
    excel_df = pd.concat(dataframes, ignore_index=True)
    print("All Excel files combined.")

    # ================== 🔧 CLEAN NUMERIC COLUMNS ==================
    def to_int_safe(series):
        return (
            pd.to_numeric(series, errors='coerce')
            .fillna(0)
            .round(0)
            .astype('int64')
        )

    excel_df['Declare line'] = to_int_safe(excel_df['Declare line'])

    # ================== 🔗 MAP from pdf_df ==================
    pdf_index = pdf_df.set_index('Matched Data')

    excel_df['Page Count'] = excel_df['Import Entry No.'].map(
        pdf_index['Page Count']
    )
    excel_df['Page Count'] = to_int_safe(excel_df['Page Count'])

    # ================== 🆕 MAP Pattern Type from Original Pattern ==================
    import re

    def detect_pattern_type(original_pattern):
        """
        return 'with_sep' or 'no_sep' by pattern that found in the PDF
        with_sep → A013 0-6708-13029  : 1st page 2 items, 5 remaining
        no_sep   → A0030680210966     : 1st page 3 items, 6 remaining
        """
        if pd.isna(original_pattern) or original_pattern is None:
            return 'with_sep'  # default
        if re.match(r'A0\d{1,2}\s\d{1,4}-\d{1,4}-\d{1,5}', str(original_pattern)):
            return 'with_sep'
        elif re.match(r'A0\d{2}\d{10,13}', str(original_pattern)):
            return 'no_sep'
        return 'with_sep'  # default

    # Map Original Pattern → Pattern Type
    excel_df['Pattern Type'] = excel_df['Import Entry No.'].map(
        pdf_index['Original Pattern']
    ).apply(detect_pattern_type)

    # ================== 📄 CALCULATE PAGE POSITION (Seperated by Pattern Type) ==================
    def calc_page_position(row):
        declare_line = row['Declare line']
        pattern_type = row['Pattern Type']

        if pattern_type == 'no_sep':
            # 1st page 3 items, 6 remaining
            if declare_line <= 3:
                return 1
            else:
                return ((declare_line - 4) // 6) + 2
        else:
            # 1st page 2 items, 5 remaining (with_sep / default)
            if declare_line <= 2:
                return 1
            else:
                return ((declare_line - 3) // 5) + 2

    excel_df['Page Position'] = excel_df.apply(calc_page_position, axis=1).astype('int64')

    # ================== 🖨 CREATE PAGE PRINT STRING ==================
    last_page = (excel_df['Page Count'] - 1).clip(lower=1)
    excel_df['Page Print'] = (
        "1,"
        + excel_df['Page Position'].astype(str)
        + ","
        + last_page.astype(str)
    )

    # ================== ✂ REMOVE CONSECUTIVE DUPLICATE PAGES ==================
    def remove_consecutive_duplicates(page_str):
        pages = page_str.split(',')
        cleaned = [pages[0]]
        for p in pages[1:]:
            if p != cleaned[-1]:
                cleaned.append(p)
        return ",".join(cleaned)

    excel_df['Page Print'] = excel_df['Page Print'].apply(remove_consecutive_duplicates)

    # ================== ✅ RESULT ==================
    print(excel_df[[
        'Import Entry No.', 'Declare line', 'Pattern Type',
        'Page Count', 'Page Position', 'Page Print'
    ]].head(10))

    end_time = time.time()
    print(f"Execution time: {end_time - start_time:.2f} seconds")

else:
    print("No Excel files found.")

4. Merge PDF -> Convert to combined_output.pdf

In [ ]:
# Folder paths
import_entry_path = r"<YOUR_PATH>" # Import Entry (PDF) folder path
import_invoice_path = r"<YOUR_PATH>" # Invoice (PDF) folder path
output_folder = r"<YOUR_PATH>" # combined pdf To print
os.makedirs(output_folder, exist_ok=True)
output_pdf_path = os.path.join(output_folder, "combined_output.pdf")

# Start the timer
start_time = time.time()

# New document to combine everything
final_pdf = fitz.open()

# Loop through each row in the Excel data
for index, row in excel_df.iterrows():
    import_entry = str(row['Import Entry No.']).strip()
    print(f"[{index+1}/{len(excel_df)}] Processing: {import_entry}")
    page_print = str(row['Page Print']).strip().replace(" ", "")
    invoice_name = str(row['Invoice No.']).strip() #Invoice No.>>

    # --- 1. Add Import Entry (specific pages) ---
    entry_pdf_path = os.path.join(import_entry_path, f"{import_entry}.pdf")
    if os.path.isfile(entry_pdf_path):
        try:
            entry_pdf = fitz.open(entry_pdf_path)
            page_numbers = [int(p.strip()) - 1 for p in page_print.split(",") if p.strip().isdigit()]
            for p in page_numbers:
                if 0 <= p < len(entry_pdf):
                    final_pdf.insert_pdf(entry_pdf, from_page=p, to_page=p)
                else:
                    print(f"[⚠️] Page {p+1} out of range for {import_entry}")
            entry_pdf.close()
        except Exception as e:
            print(f"[💥] Error adding Import Entry {import_entry}: {e}")
    else:
        print(f"[❌] Import Entry file not found: {entry_pdf_path}")

    # --- 2. Add Import Invoice (entire file) ---
    invoice_pdf_path = os.path.join(import_invoice_path, f"{invoice_name}.pdf")
    if os.path.isfile(invoice_pdf_path):
        try:
            invoice_pdf = fitz.open(invoice_pdf_path)
            final_pdf.insert_pdf(invoice_pdf)
            invoice_pdf.close()
        except Exception as e:
            print(f"[💥] Error adding invoice {invoice_name}: {e}")
    else:
        print(f"[❌] Invoice file not found: {invoice_pdf_path}")

# End the timer
end_time = time.time()

# Save the combined file
final_pdf.save(output_pdf_path)
final_pdf.close()

print(f"\n✅ Combined PDF saved to: {output_pdf_path}")
# Calculate total execution time
execution_time = end_time - start_time
print(f"Total execution time: {execution_time:.2f} seconds")

Attempt 1 (failed): Highlight whole PDF block // Pattern PDF 1 (1st page 2 items, 5 remaining) -> Save Highlighted PDF for each row item

In [ ]:
import os
import pandas as pd
import fitz  # PyMuPDF

# --- Folder paths ---
import_entry_path = r"<YOUR_PATH>" # Import Entry (PDF) folder path
import_invoice_path = r"<YOUR_PATH>" # Invoice (PDF) folder path
output_folder = r"<YOUR_PATH>" # combined pdf To print
os.makedirs(output_folder, exist_ok=True)
output_pdf_path = os.path.join(output_folder, "combined_output.pdf")

# --- Master block lists ---
first_page_items = [
    [94, 109],   # page 1
    [110, 125]   # page 2
]

other_page_items = [
    [4, 24],
    [25, 40],
    [41, 56],
    [57, 72],
    [73, 88]
]

def get_blocks_to_highlight(page_position):
    """Return correct blocks to highlight based on page_position"""
    if page_position <= 2:
        return first_page_items[page_position - 1]
    else:
        item_index = (page_position - 3) % 5
        return other_page_items[item_index]

def highlight_page(pdf_path, page_pos):
    """
    Open a PDF, extract blocks, highlight selected blocks on the given page,
    save to a new file, and return the new file path.
    """
    doc = fitz.open(pdf_path)
    page_index = page_pos - 1  # Excel is 1-based, fitz is 0-based
    if page_index >= len(doc):
        print(f"[⚠️] Page {page_pos} out of range for {pdf_path}")
        doc.close()
        return None

    page = doc[page_index]

    # Extract blocks
    blocks = page.get_text("blocks")
    print(f"\n🔹 Processing: {os.path.basename(pdf_path)} | Page: {page_pos} | Total blocks: {len(blocks)}")

    # Get block indices
    block_indices = get_blocks_to_highlight(page_pos)

    # Highlight
    for b_idx in block_indices:
        if 0 <= b_idx-1 < len(blocks):
            x0, y0, x1, y1, *_ = blocks[b_idx-1]
            page.draw_rect(
                fitz.Rect(x0, y0, x1, y1),
                fill=(1, 1, 0),
                fill_opacity=0.3,
                color=None,
                width=0,
                overlay=True
            )

    # Save to new file
    output_path = pdf_path.replace(".pdf", f"_{page_pos}_{len(blocks)}.pdf")
    doc.save(output_path)
    doc.close()
    print(f"✅ Highlighted page {page_pos} saved → {output_path}")
    return output_path

# --- Final combined PDF ---
final_pdf = fitz.open()

# --- Loop through Excel rows ---
for idx, row in excel_df.iterrows():
    import_entry = str(row['Import Entry No.']).strip()
    page_print = str(row['Page Print']).strip().replace(" ", "")
    invoice_name = str(row['Invoice No.']).strip()

    # --- 1. Import Entry ---
    entry_pdf_path = os.path.join(import_entry_path, f"{import_entry}.pdf")
    if os.path.isfile(entry_pdf_path):
        try:
            page_numbers = [int(p.strip()) for p in page_print.split(",") if p.strip().isdigit()]
            for page_pos in page_numbers:
                highlighted_path = highlight_page(entry_pdf_path, page_pos)
                if highlighted_path and os.path.isfile(highlighted_path):
                    temp_pdf = fitz.open(highlighted_path)
                    final_pdf.insert_pdf(temp_pdf)
                    temp_pdf.close()
        except Exception as e:
            print(f"[💥] Error processing Import Entry {import_entry}: {e}")
    else:
        print(f"[❌] Import Entry not found: {entry_pdf_path}")

    # --- 2. Import Invoice ---
    invoice_pdf_path = os.path.join(import_invoice_path, f"{invoice_name}.pdf")
    if os.path.isfile(invoice_pdf_path):
        try:
            invoice_pdf = fitz.open(invoice_pdf_path)
            final_pdf.insert_pdf(invoice_pdf)
            invoice_pdf.close()
        except Exception as e:
            print(f"[💥] Error adding invoice {invoice_name}: {e}")
    else:
        print(f"[❌] Invoice not found: {invoice_pdf_path}")

# --- Save combined PDF ---
final_pdf.save(output_pdf_path)
final_pdf.close()
print(f"\n✅ Combined PDF saved with highlights: {output_pdf_path}")